# trianglebench – reproducible analysis

This notebook (1) optionally runs `trianglebench` on the GPU and `trianglebench-cpu` on the CPU of this container, (2) loads every result file
(reference data shipped with the image plus your own runs in `results/`), and (3) analyses them:

1. provenance and correctness (all methods agree node by node, identical triangle counts across systems)
2. descriptive statistics per system, graph size and method (mean, 95 % CI, CV)
3. performance in relation to **power, energy, temperature and clock**
4. raw telemetry over time (power, temperature, SM clock, throttle reasons)
5. inference: Welch t-tests between systems with Holm correction, effect sizes; CPU vs. GPU for the same algorithm
6. design checks: order effects and drift over repetitions (log-linear OLS)
7. export of tables, figures and the software environment to `export/` for sharing

Unit of analysis is one **measuring window** (one method, one repetition); solves inside a window are
autocorrelated and are not treated as independent samples.

**Settings** (environment variables, or edit the next cell):
`TB_RUN=1` runs a new GPU benchmark, `TB_RUN_CPU=1` the CPU-only benchmark, `TB_ARGS` / `TB_CPU_ARGS` set their options,
`TB_LABEL` describes the system.

In [ ]:
%matplotlib inline
import os, sys, glob, json, shlex, platform, subprocess, itertools, datetime
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib, statsmodels
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

TB_HOME = Path(os.environ.get("TB_HOME", "/opt/trianglebench"))
if not (TB_HOME / "analysis").exists():          # notebook opened inside a git checkout
    TB_HOME = Path.cwd().resolve().parent
WORK = Path.cwd()
RESULTS = WORK / "results"; RESULTS.mkdir(exist_ok=True)
EXPORT = WORK / "export"; EXPORT.mkdir(exist_ok=True)
sys.path.insert(0, str(TB_HOME / "analysis"))
import summarize

RUN_BENCHMARK = os.environ.get("TB_RUN", "0") == "1"
BENCH_ARGS = os.environ.get("TB_ARGS", "--n 4096,8192 --repeats 10 --seconds 5 --warmup 10")
LABEL = os.environ.get("TB_LABEL", "JupyterLab container")
RUN_CPU = os.environ.get("TB_RUN_CPU", "0") == "1"
CPU_ARGS = os.environ.get("TB_CPU_ARGS", "--n 4096,8192 --repeats 10 --seconds 5 --warmup 10")
ALPHA = 0.05
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
print(f"python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__} | "
      f"statsmodels {statsmodels.__version__} | matplotlib {matplotlib.__version__}")
print("TB_HOME", TB_HOME, "| WORK", WORK, "| RUN_BENCHMARK", RUN_BENCHMARK, "| RUN_CPU", RUN_CPU)

## 1. Optional: run the benchmarks here

Each program runs its self-test first (brute-force count vs. all methods), then the benchmark.
The reference protocol (`--n 4096,8192 --repeats 10 --seconds 5 --warmup 10`) takes about 9 minutes on the GPU
and about 5 minutes on the CPU. The measured device should be otherwise idle; describe any other load in `TB_LABEL`.
CPU power needs RAPL (x86) with read access to `/sys/class/powercap` (usually root); without it watts stay empty.


In [ ]:
def run_bench(binary, args, tag):
    st = subprocess.run([binary, "--selftest"], capture_output=True, text=True)
    print(st.stdout[-600:])
    assert st.returncode == 0, f"{binary} self-test failed - do not use this system for measurements"
    prefix = RESULTS / (tag + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    cmd = [binary, *shlex.split(args), "--label", LABEL, "--out", str(prefix)]
    print(shlex.join(cmd))
    run = subprocess.run(cmd, capture_output=True, text=True)
    print(run.stdout[-2500:], run.stderr[-1500:])
    assert run.returncode == 0, "benchmark failed or methods disagree"

if RUN_BENCHMARK:
    run_bench("trianglebench", BENCH_ARGS, "tb_")
if RUN_CPU:
    run_bench("trianglebench-cpu", CPU_ARGS, "tbcpu_")
if not (RUN_BENCHMARK or RUN_CPU):
    print("TB_RUN / TB_RUN_CPU are off - analysing existing result files only")

## 2. Load all result files and check provenance

In [ ]:
patterns = [TB_HOME / "results" / "*.json", RESULTS / "*.json", WORK / "*.json"]
files = sorted({str(Path(f).resolve()) for p in patterns for f in glob.glob(str(p))})
meta, frames, seen = [], [], set()
for f in files:
    d = json.load(open(f))
    schema = str(d.get("schema", ""))
    if schema.startswith("trianglebench-result/"):
        g = d["gpu"]
        info = {"device_type": "gpu", "device": g["name"], "id": g["uuid"], "cc": g["compute_capability"], "driver": g["driver"],
                "cuda_runtime": g["cuda_runtime"], "cublaslt": g["cublaslt"], "power_limit_w": g["power_limit_w"],
                "integrated": g["integrated"], "threads": None}
    elif schema.startswith("trianglebench-cpu-result/"):
        c = d["cpu"]
        info = {"device_type": "cpu", "device": c["name"], "id": c["name"] + "|" + d["system"]["product"], "cc": d["tool"]["build_arch"],
                "driver": c["cpufreq_driver"], "cuda_runtime": None, "cublaslt": d["tool"]["blas"], "power_limit_w": None,
                "integrated": None, "threads": c["openmp_threads"]}
    else:
        continue
    key = (info["id"], d["start_utc"])
    if key in seen:                                   # same run stored twice
        continue
    seen.add(key)
    meta.append({"file": f, "label": d["label"], **info, "machine": d["system"]["machine"], "kernel": d["system"]["kernel"],
                 "tool": d["tool"]["version"], "commit": d["tool"]["git_commit"], "schema": schema, "start_utc": d["start_utc"],
                 "energy_counter": d["telemetry"]["energy_counter"], "all_verified": d["all_verified"],
                 "telemetry_file": str(Path(f).with_name(Path(d["telemetry"]["file"]).name)), "json": d})
    frames.append(pd.DataFrame(summarize.rows(f)))
assert meta, "no result files found"
meta = pd.DataFrame(meta)
# readable, unique system name: device (+ label if one device appears in several files)
meta["system"] = np.where(meta["device_type"] == "cpu", "CPU " + meta["device"], meta["device"])
dup = meta["system"].duplicated(keep=False)
meta.loc[dup, "system"] = meta.loc[dup, "system"] + " | " + meta.loc[dup, "label"]
df = pd.concat(frames, ignore_index=True).merge(meta[["file", "system"]], on="file")
NUMERIC = ["eq_tops", "seconds_per_solve", "watts", "energy_j", "joule_per_solve", "eq_tops_per_watt", "temp_max_c", "sm_mhz_median"]
df[NUMERIC] = df[NUMERIC].apply(pd.to_numeric, errors="coerce")   # missing sensors (None) -> NaN
df["kind"] = np.where(df["method"] == "P_all_four_parallel", "parallel", "alone")
SHORT = {"M1_cuda_int32_bitset_popc": "M1 CUDA INT32 bitset", "M2_cuda_fp32_gemm": "M2 CUDA FP32 GEMM",
         "M3_tensor_fp16_fp32acc_gemm": "M3 Tensor FP16", "M4_tensor_int8_int32acc_gemm": "M4 Tensor INT8",
         "P_all_four_parallel": "all four parallel",
         "C1_cpu_int64_bitset_popc": "C1 CPU INT64 bitset", "C2_cpu_fp32_gemm_blas": "C2 CPU FP32 GEMM"}
df["method_short"] = df["method"].map(SHORT)
SYSTEMS = list(meta["system"])
print(f"{len(meta)} result files ({(meta['device_type'] == 'gpu').sum()} GPU, {(meta['device_type'] == 'cpu').sum()} CPU), "
      f"{len(df)} measuring windows")
meta.drop(columns=["json", "file", "telemetry_file", "id"])

### Correctness and reproducibility

Every method must agree with every other one on every node (`all_verified`), and the same `(n, p, seed)`
must give the same triangle count on every system, independent of GPU and CPU architecture -
CPU and GPU programs share the graph generator, so the CPU runs are an independent cross-check of the GPU runs.

In [ ]:
tri = []
for _, m in meta.iterrows():
    par = m["json"]["parameters"]
    for r in m["json"]["runs"]:
        tri.append({"system": m["system"], "n": r["n"], "p": par["p"], "seed": par["seed"], "triangles": r["graph"]["triangles"],
                    "ratio_to_theory": r["graph"]["triangles"] / r["graph"]["triangles_expected"],
                    "global_clustering": r["graph"]["global_clustering_fp64"], "verified": m["all_verified"]})
tri = pd.DataFrame(tri)
consistent = tri.groupby(["n", "p", "seed"])["triangles"].nunique().eq(1)
assert tri["verified"].all(), "a result file contains a mismatch between methods"
assert consistent.all(), "different triangle counts for the same graph"
print("all files verified, triangle counts identical across systems for", len(consistent), "graph(s)")
tri.pivot_table(index=["n", "p", "seed"], columns="system", values="triangles")

## 3. Descriptive statistics (95 % CI from Student t)

In [ ]:
def ci95(x):
    x = pd.Series(x).dropna()
    return stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

summary = (df.groupby(["system", "n", "method_short"])
             .agg(k=("eq_tops", "size"), eq_tops=("eq_tops", "mean"), eq_tops_ci95=("eq_tops", ci95),
                  cv_pct=("eq_tops", lambda x: 100 * x.std(ddof=1) / x.mean()), watts=("watts", "mean"),
                  tops_per_w=("eq_tops_per_watt", "mean"), tops_per_w_ci95=("eq_tops_per_watt", ci95),
                  joule_per_solve=("joule_per_solve", "mean"), temp_max_c=("temp_max_c", "max"),
                  sm_mhz=("sm_mhz_median", "median"))
             .round(4))
summary.to_csv(EXPORT / "summary.csv")
summary

## 4. Performance in relation to power, energy, temperature and clock

In [ ]:
ORDER = list(SHORT.values())
COLORS = dict(zip(SYSTEMS, plt.rcParams["axes.prop_cycle"].by_key()["color"] * 5))
def bars(metric, err, ylabel, fname, data=summary):
    sizes = sorted(df["n"].unique())
    ncol = min(3, len(sizes)); nrow = -(-len(sizes) // ncol)
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.5 * ncol, 4.4 * nrow + 0.6), squeeze=False)
    for ax in axes.flat[len(sizes):]:
        ax.set_visible(False)
    for ax, n in zip(axes.flat, sizes):
        s = data.xs(n, level="n")
        methods = [m for m in ORDER if m in s.index.get_level_values("method_short")]
        w = 0.8 / max(1, len(SYSTEMS))
        for i, sysname in enumerate(SYSTEMS):
            if sysname not in s.index.get_level_values("system"):
                continue
            t = s.loc[sysname].reindex(methods)
            ax.bar(np.arange(len(methods)) + i * w, t[metric], w, yerr=t[err] if err else None, capsize=3,
                   color=COLORS[sysname])
        ax.set_xticks(np.arange(len(methods)) + w * (len(SYSTEMS) - 1) / 2, methods, rotation=25, ha="right")
        vals = s[metric].replace(0, np.nan).dropna()
        if len(vals) and vals.max() / vals.min() > 30:  # GPU and CPU on one axis: log scale keeps both readable
            ax.set_yscale("log")
        ax.set_title(f"n = {n}" + (" (log scale)" if ax.get_yscale() == "log" else "")); ax.set_ylabel(ylabel); ax.grid(axis="y", alpha=.3)
    fig.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLORS[s]) for s in SYSTEMS], labels=SYSTEMS,
               loc="upper center", ncol=min(3, len(SYSTEMS)), fontsize=8, frameon=False)
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.5 / fig.get_figheight()))
    for ext in ("png", "svg"):
        fig.savefig(EXPORT / f"{fname}.{ext}", dpi=150)
    plt.show()

bars("eq_tops", "eq_tops_ci95", "equivalent TOPS (2n³ / time), mean ± 95 % CI", "fig1_throughput")
bars("tops_per_w", "tops_per_w_ci95", "equivalent TOPS per watt", "fig2_efficiency")
bars("joule_per_solve", None, "joule per complete triangle count", "fig3_energy_per_solution")

In [ ]:
alone = df[df["kind"] == "alone"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
markers = dict(zip(ORDER, "osD^*vP"))
colors = COLORS
for (sysname, meth), g in df.groupby(["system", "method_short"]):
    kw = dict(color=colors[sysname], marker=markers[meth], alpha=.7, s=28)
    axes[0].scatter(g["watts"], g["eq_tops"], **kw)
    axes[1].scatter(g["temp_max_c"], g["eq_tops_per_watt"], **kw)
    axes[2].scatter(g["sm_mhz_median"], g["eq_tops"], **kw)
for ax, (x, y) in zip(axes, [("power [W] (GPU: board/module, CPU: RAPL package)", "eq.TOPS"), ("max. device temperature [°C]", "eq.TOPS per watt"),
                             ("median clock [MHz] (GPU: SM, CPU: core)", "eq.TOPS")]):
    ax.set_xlabel(x); ax.set_ylabel(y); ax.grid(alpha=.3)
    yv = [c.get_offsets()[:, 1] for c in ax.collections]
    yv = np.concatenate(yv) if yv else np.array([])
    yv = yv[np.isfinite(yv) & (yv > 0)]
    if len(yv) and yv.max() / yv.min() > 30:
        ax.set_yscale("log")
handles = [plt.Line2D([], [], color=c, marker="o", ls="", label=s) for s, c in colors.items()] + \
          [plt.Line2D([], [], color="gray", marker=mk, ls="", label=m) for m, mk in markers.items()]
axes[0].legend(handles=handles, fontsize=7)
axes[0].set_title("throughput vs. power (one point = one window)"); axes[1].set_title("efficiency vs. temperature"); axes[2].set_title("throughput vs. clock")
fig.tight_layout(); fig.savefig(EXPORT / "fig4_relations.png", dpi=150); fig.savefig(EXPORT / "fig4_relations.svg"); plt.show()

## 5. Raw telemetry over time

Measuring windows are placed one after another on a common time axis (gaps between windows removed).
NVML clock-event reasons are decoded for GPUs; on integrated GPUs (GB10) several NVML fields are not reported.
CPU files contain RAPL package power (empty without RAPL), the hottest temperature sensor, the median core clock
and the package thermal-throttle counter.

In [ ]:
REASONS = {0x1: "gpu_idle", 0x2: "app_clock_setting", 0x4: "sw_power_cap", 0x8: "hw_slowdown", 0x20: "sw_thermal",
           0x40: "hw_thermal", 0x80: "hw_power_brake", 0x100: "display_clock"}
tel = []
for _, m in meta.iterrows():
    if not Path(m["telemetry_file"]).exists():
        print("missing telemetry:", m["telemetry_file"]); continue
    t = pd.read_csv(m["telemetry_file"])
    t["system"] = m["system"]
    win = (t[["n", "phase", "method", "repeat"]] != t[["n", "phase", "method", "repeat"]].shift()).any(axis=1).cumsum()
    t["window"] = win
    offset = t.groupby("window")["t_s"].max().cumsum().shift(fill_value=0)
    t["timeline_s"] = t["t_s"] + t["window"].map(offset)
    if "cpu_mhz" in t:                                # CPU telemetry: same columns as far as they exist
        t = t.rename(columns={"cpu_mhz": "sm_mhz"})
        t["reasons"] = 0
    else:
        t["reasons"] = t["clock_event_reasons"].map(lambda v: int(str(v), 16))
    t["device_type"] = m["device_type"]
    tel.append(t)
tel = pd.concat(tel, ignore_index=True)
fig, axes = plt.subplots(3, 1, figsize=(15, 8), sharex=False)
for sysname, t in tel.groupby("system"):
    for ax, col in zip(axes, ["power_w", "temp_c", "sm_mhz"]):
        ax.plot(t["timeline_s"], t[col], lw=.6, label=sysname, color=colors.get(sysname))
for ax, y in zip(axes, ["power [W]", "temperature [°C]", "clock [MHz] (GPU SM / CPU core)"]):
    ax.set_ylabel(y); ax.grid(alpha=.3)
axes[0].legend(fontsize=8); axes[2].set_xlabel("measuring time [s] (windows concatenated)")
fig.tight_layout(); fig.savefig(EXPORT / "fig5_telemetry.png", dpi=150); plt.show()

share = {s: {name: round(100 * (g["reasons"] & bit).astype(bool).mean(), 1) for bit, name in REASONS.items()}
         for s, g in tel[(tel["phase"] == "alone") & (tel["device_type"] == "gpu")].groupby("system")}
pd.DataFrame(share).rename_axis("clock-event reason [% of samples]")

## 6. Inference: systems compared per graph size and method

Welch t-test on per-window eq.TOPS and eq.TOPS/W, Holm correction over all tests, Hedges' g as effect size.
Ratios are `B / A`. With only one system loaded this section is skipped.

In [ ]:
def hedges_g(a, b):
    na, nb = len(a), len(b)
    sp = np.sqrt(((na - 1) * np.var(a, ddof=1) + (nb - 1) * np.var(b, ddof=1)) / (na + nb - 2))
    return (np.mean(b) - np.mean(a)) / sp * (1 - 3 / (4 * (na + nb) - 9))

tests = []
for A, B in itertools.combinations(SYSTEMS, 2):
    for (n, meth), ga in df[df["system"] == A].groupby(["n", "method_short"]):
        gb = df[(df["system"] == B) & (df["n"] == n) & (df["method_short"] == meth)]
        if len(ga) < 2 or len(gb) < 2:
            continue
        for metric in ("eq_tops", "eq_tops_per_watt"):
            a, b = ga[metric].dropna().to_numpy(), gb[metric].dropna().to_numpy()
            if len(a) < 2 or len(b) < 2:                   # e.g. no CPU energy counter on this platform
                continue
            t, p = stats.ttest_ind(b, a, equal_var=False)
            tests.append({"A": A, "B": B, "n": n, "method": meth, "metric": metric, "ratio_B_over_A": b.mean() / a.mean(),
                          "t": t, "p": p, "hedges_g": hedges_g(a, b), "k_A": len(a), "k_B": len(b)})
if tests:
    tests = pd.DataFrame(tests)
    tests["p_holm"] = multipletests(tests["p"], method="holm")[1]
    tests["significant"] = tests["p_holm"] < ALPHA
    tests.to_csv(EXPORT / "tests_between_systems.csv", index=False)
    display(tests.round({"ratio_B_over_A": 3, "t": 2, "hedges_g": 2}))
else:
    print("only one system loaded - nothing to compare")

### CPU vs. GPU for the same algorithm

`M1` and `C1` count with bitsets and popcount, `M2` and `C2` multiply FP32 matrices; "best" compares the fastest
method of each device. Speed-up = GPU / CPU with a 95 % CI from a Welch interval on log(eq.TOPS);
`energy_ratio` = CPU joule per solve / GPU joule per solve (only where both devices have an energy counter).


In [ ]:
def ratio_ci(gpu, cpu):
    """GPU/CPU ratio of means with 95 % CI from a Welch interval on the log scale."""
    a, b = np.log(cpu), np.log(gpu)
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    dfw = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    h = stats.t.ppf(0.975, dfw) * np.sqrt(va + vb)
    d = b.mean() - a.mean()
    return np.exp(d), np.exp(d - h), np.exp(d + h)

pairs = [("M1 CUDA INT32 bitset", "C1 CPU INT64 bitset"), ("M2 CUDA FP32 GEMM", "C2 CPU FP32 GEMM"), ("best", "best")]
gpus = meta.loc[meta["device_type"] == "gpu", "system"]; cpus = meta.loc[meta["device_type"] == "cpu", "system"]
xr = []
for G, Cs in itertools.product(gpus, cpus):
    for n in sorted(set(df.loc[df["system"] == G, "n"]) & set(df.loc[df["system"] == Cs, "n"])):
        a_ = alone[(alone["system"] == G) & (alone["n"] == n)]; c_ = df[(df["system"] == Cs) & (df["n"] == n)]
        for mg, mc in pairs:
            if mg == "best":
                mg = a_.groupby("method_short")["eq_tops"].mean().idxmax(); mc = c_.groupby("method_short")["eq_tops"].mean().idxmax()
            g, c = a_[a_["method_short"] == mg], c_[c_["method_short"] == mc]
            if len(g) < 2 or len(c) < 2:
                continue
            r, lo, hi = ratio_ci(g["eq_tops"].to_numpy(), c["eq_tops"].to_numpy())
            jg, jc = g["joule_per_solve"].mean(), c["joule_per_solve"].mean()
            xr.append({"GPU": G, "CPU": Cs, "n": n, "gpu_method": mg, "cpu_method": mc, "gpu_eq_tops": g["eq_tops"].mean(),
                       "cpu_eq_tops": c["eq_tops"].mean(), "speedup_gpu_over_cpu": r, "ci95_low": lo, "ci95_high": hi,
                       "gpu_watts": g["watts"].mean(), "cpu_watts": c["watts"].mean(), "gpu_j_per_solve": jg, "cpu_j_per_solve": jc,
                       "energy_ratio_cpu_over_gpu": jc / jg if np.isfinite(jc) and np.isfinite(jg) and jg else np.nan})
if xr:
    xr = pd.DataFrame(xr)
    xr.to_csv(EXPORT / "cpu_vs_gpu.csv", index=False)
    display(xr.round(3))
else:
    print("load at least one GPU and one CPU result file for this comparison")

## 7. Design checks: order effects and drift

The method order rotates every repetition. A log-linear model per system and graph size,
`log(eq_tops) ~ C(method) + position_in_order` (and `+ repeat` for drift), estimates the relative change per
step in percent. A significant effect means warm-up or heat build-up influenced the comparison.

In [ ]:
rows_ = []
for (sysname, n), g in alone.groupby(["system", "n"]):
    g = g.assign(log_tops=np.log(g["eq_tops"]))
    for term in ("position_in_order", "repeat"):
        fit = smf.ols(f"log_tops ~ C(method) + {term}", data=g).fit()
        lo, hi = fit.conf_int().loc[term]
        rows_.append({"system": sysname, "n": n, "term": term, "pct_per_step": 100 * np.expm1(fit.params[term]),
                      "ci95_low_pct": 100 * np.expm1(lo), "ci95_high_pct": 100 * np.expm1(hi), "p": fit.pvalues[term], "windows": len(g)})
design = pd.DataFrame(rows_)
design.to_csv(EXPORT / "design_checks.csv", index=False)
design.round(4)

## 8. Parallel execution

`sum_of_shares` = Σ (throughput of a method in parallel / its throughput alone). 1 means the four methods
only share the GPU; values above 1 would mean real gains from running CUDA cores and Tensor Cores at once.

In [ ]:
par = []
for _, m in meta.iterrows():
    for r in m["json"]["runs"]:
        if "parallel" not in r:                       # CPU files have no parallel phase
            continue
        s = r["parallel"]["summary"]["sum_of_shares"]
        par.append({"system": m["system"], "n": r["n"], "sum_of_shares": s["mean"], "ci95": s["ci95_halfwidth"], "k": s["k"],
                    "eq_tops_parallel": r["parallel"]["summary"]["eq_tops"]["mean"]})
pd.DataFrame(par).round(4)

## 9. Export for sharing

`export/` now contains tables (CSV), figures (PNG, SVG) and `environment.json`. Share the executed notebook as HTML:
*File → Save and Export Notebook As → HTML*, or in a terminal:

```
jupyter nbconvert --to html --execute trianglebench-analysis.ipynb --output-dir export
```

When citing results, give the tool version, the git commit and the schema stored in every result file.

In [ ]:
df.to_csv(EXPORT / "all_windows_tidy.csv", index=False)
tel.to_csv(EXPORT / "telemetry_all.csv", index=False)
env = {"created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
       "python": platform.python_version(), "platform": platform.platform(),
       "packages": {"numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
                    "statsmodels": statsmodels.__version__, "matplotlib": matplotlib.__version__},
       "result_files": [{"file": Path(f).name, "tool": t, "commit": c, "schema": s}
                        for f, t, c, s in zip(meta["file"], meta["tool"], meta["commit"], meta["schema"])]}
json.dump(env, open(EXPORT / "environment.json", "w"), indent=2)
print(sorted(p.name for p in EXPORT.iterdir()))